# 6 — Ablation: Base-Rate Check

The freq=10 canary (secret `116632`) shows ~4 bits exposure even under strong DP,
while freq=50 (`791772`) is near-random (~0.7 bits). Since each frequency uses a
**different** secret, this could reflect a base-rate bias in GPT-2 rather than
genuine memorization.

This notebook scores all 4 canary secrets on **pretrained GPT-2** (no fine-tuning)
to measure their a priori rank. If `116632` already ranks high before any training,
the anomaly is a measurement artifact.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/UofT/CSC2412/dp-finetuned-llms'
print('Drive mounted. Project dir:', PROJECT_DIR)

## 2. Install Dependencies

In [ ]:
!pip install -q transformers matplotlib

## 3. Imports & Config

In [ ]:
import os
import json
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

CANARY_PREFIX   = 'My patient ID is '
SCORE_BATCH     = 256
MAX_SEQ_LEN     = 32
N_SECRETS       = 900_000

CANARIES_PATH   = f'{PROJECT_DIR}/data/canaries.json'
EXPOSURE_PATH   = f'{PROJECT_DIR}/results/exposure_results.json'
BASE_RATE_PATH  = f'{PROJECT_DIR}/results/base_rate.json'
PLOTS_DIR       = f'{PROJECT_DIR}/results/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 4. Load Canary Metadata & DP Exposure Results

In [ ]:
with open(CANARIES_PATH) as f:
    canaries = json.load(f)

with open(EXPOSURE_PATH) as f:
    exposure_results = json.load(f)

print('Canaries:')
for c in canaries:
    print(f"  freq={c['freq']:>2}x  secret={c['secret']}")

## 5. Build Candidate List

In [ ]:
all_secrets = [str(n) for n in range(100_000, 1_000_000)]
print(f'Candidate pool: {len(all_secrets):,}')

## 6. Scoring Helper

In [ ]:
def score_candidates(model, tokenizer, prefix, candidates,
                     batch_size=SCORE_BATCH, device=device):
    """Avg per-token log-prob of (prefix + candidate) for each candidate."""
    model.eval()
    all_scores = []
    with torch.no_grad():
        for i in tqdm(range(0, len(candidates), batch_size),
                      desc='scoring', leave=False):
            batch      = candidates[i : i + batch_size]
            full_texts = [prefix + c for c in batch]
            enc = tokenizer(
                full_texts, return_tensors='pt',
                padding=True, truncation=True, max_length=MAX_SEQ_LEN,
            )
            input_ids = enc['input_ids'].to(device)
            attn_mask = enc['attention_mask'].to(device)
            outputs      = model(input_ids=input_ids, attention_mask=attn_mask)
            shift_logits = outputs.logits[:, :-1, :].float()
            shift_labels = input_ids[:, 1:]
            tok_lp = (torch.log_softmax(shift_logits, dim=-1)
                      .gather(2, shift_labels.unsqueeze(-1))
                      .squeeze(-1))
            pad_mask = (shift_labels != tokenizer.pad_token_id).float()
            n_tok    = pad_mask.sum(dim=-1).clamp(min=1)
            score    = (tok_lp * pad_mask).sum(dim=-1) / n_tok
            all_scores.extend(score.cpu().tolist())
    return np.array(all_scores)


def compute_exposure(scores, true_secret, all_secrets):
    true_idx   = all_secrets.index(true_secret)
    true_score = scores[true_idx]
    rank       = int((scores > true_score).sum()) + 1
    exposure   = math.log2(len(all_secrets)) - math.log2(rank)
    return rank, exposure, float(true_score)


print('Helpers defined.')

## 7. Score on Pretrained GPT-2

Load vanilla `gpt2` (no fine-tuning, no Conv1D replacement) and score all
900 000 candidates.  This takes ~30 s on T4.

In [ ]:
# Use the project tokenizer (same vocab size as fine-tuned models)
tokenizer = GPT2TokenizerFast.from_pretrained(
    f'{PROJECT_DIR}/data/tokenizer')

# Vanilla pretrained GPT-2 — resize embeddings to match project tokenizer
base_model = GPT2LMHeadModel.from_pretrained('gpt2')
base_model.resize_token_embeddings(len(tokenizer))
base_model = base_model.half().to(device).eval()

print('Model loaded. Scoring 900k candidates...')
base_scores = score_candidates(base_model, tokenizer, CANARY_PREFIX, all_secrets)
print('Done.')

## 8. Base-Rate Rank & Exposure for Each Canary Secret

In [ ]:
base_rate_results = []

print(f'{"freq":>5}x  {"secret":>8}  {"rank":>10}  {"exposure":>9}  {"log_prob":>10}')
print('-' * 55)

for c in canaries:
    rank, exposure, lp = compute_exposure(base_scores, c['secret'], all_secrets)
    base_rate_results.append({
        'freq':     c['freq'],
        'secret':   c['secret'],
        'rank':     rank,
        'exposure': exposure,
        'log_prob': lp,
    })
    print(f"{c['freq']:>5}x  {c['secret']:>8}  {rank:>10,}  {exposure:>9.3f}  {lp:>10.4f}")

with open(BASE_RATE_PATH, 'w') as f:
    json.dump(base_rate_results, f, indent=2)
print(f'\nSaved to {BASE_RATE_PATH}')

## 9. Comparison: Base Rate vs DP-Trained Models

For each secret, compare the base-rate exposure (pretrained GPT-2) against
the exposure under each DP condition.  If the base-rate exposure for `116632`
already matches the DP exposure (~4 bits), the anomaly is a prior effect,
not memorization.

In [ ]:
EPS_ORDER  = [0.5, 1.0, 2.0, 4.0, 8.0, None]  # None = ∞
EPS_LABELS = ['0.5', '1', '2', '4', '8', '∞']

# Build lookup: (eps_target, freq) → exposure
dp_lookup = {(r['epsilon_target'], r['freq']): r['exposure']
             for r in exposure_results}
br_lookup = {r['freq']: r['exposure'] for r in base_rate_results}

print(f'{"freq":>5}x  {"base":>8}  '
      + '  '.join(f'{l:>6}' for l in EPS_LABELS))
print('-' * 70)

for c in canaries:
    freq = c['freq']
    base = br_lookup[freq]
    dp_vals = [dp_lookup.get((e, freq), float('nan')) for e in EPS_ORDER]
    row = '  '.join(f'{v:>6.2f}' for v in dp_vals)
    print(f"{freq:>5}x  {base:>8.3f}  {row}")

## 10. Plot

In [ ]:
FREQS   = [1, 5, 10, 50]
EPS_MAP = {0.5: 0.5, 1.0: 1.0, 2.0: 2.0, 4.0: 4.0, 8.0: 8.0, None: 16.0}
EPS_TICKS      = [0.5, 1, 2, 4, 8, 16]
EPS_TICK_LABELS = ['0.5', '1', '2', '4', '8', '∞']
COLORS  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
MARKERS = ['o', 's', '^', 'D']

fig, ax = plt.subplots(figsize=(7, 4.5))

for freq, color, marker in zip(FREQS, COLORS, MARKERS):
    rows = sorted([r for r in exposure_results if r['freq'] == freq],
                  key=lambda r: EPS_MAP[r['epsilon_target']])
    xs   = [EPS_MAP[r['epsilon_target']] for r in rows]
    exps = [r['exposure'] for r in rows]
    ax.plot(xs, exps, marker=marker, color=color,
            linewidth=1.8, markersize=7, label=f'freq={freq}×')
    # Base-rate dot on the left
    br = br_lookup[freq]
    ax.scatter([0.25], [br], marker=marker, color=color,
               s=60, zorder=5, edgecolors='k', linewidths=0.5)

ax.axhline(0, color='grey', linewidth=1, linestyle='--', label='random')
ax.axvline(0.4, color='lightgrey', linewidth=1, linestyle=':')
ax.text(0.27, ax.get_ylim()[1]*0.97, 'base\nrate',
        ha='center', va='top', fontsize=8, color='grey')

ax.set_xscale('log')
ax.set_xticks(EPS_TICKS)
ax.set_xticklabels(EPS_TICK_LABELS)
import matplotlib.ticker as ticker
ax.xaxis.set_minor_locator(ticker.NullLocator())
ax.set_xlabel('Privacy budget ε  (log scale)')
ax.set_ylabel('Canary exposure (bits)')
ax.set_title('Exposure vs ε — with base-rate reference', pad=10)
ax.legend(frameon=False, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/exposure_vs_epsilon_with_baserate.png',
            bbox_inches='tight')
plt.show()
print('Saved.')